# Step 4 : BN <-> EN CLIR

In [1]:
!pip install transformers sentencepiece sacremoses

In [2]:
!pip -q install torch


## 4.2 Load Saved BM25 Indexes (use Step 3 Files)

In [3]:
import pickle
import re

def load_index(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

en_pack = load_index('bm25_en.pkl')
bn_pack = load_index('bm25_bn.pkl')

bm25_en = en_pack['bm25']
doc_ids_en = en_pack['doc_ids']
docs_en = en_pack['docs']

bm25_bn = bn_pack['bm25']
doc_ids_bn = bn_pack['doc_ids']
docs_bn = bn_pack['docs']


def tokenize_en(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def tokenize_bn(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

def search_en(query, top_k=5):
    q = tokenize_en(query)
    scores = bm25_en.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_en[i], "score": float(scores[i]), "title": docs_en[i].get("title",""), "url": docs_en[i].get("url","")} for i in idx]

def search_bn(query, top_k=5):
    q = tokenize_bn(query)
    scores = bm25_bn.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_bn[i], "score": float(scores[i]), "title": docs_bn[i].get("title",""), "url": docs_bn[i].get("url","")} for i in idx]

print("Index loaded:" , len(docs_en), "English documents and", len(docs_bn), "Bengali documents.")


FileNotFoundError: [Errno 2] No such file or directory: 'bm25_en.pkl'

## 4.2 Language Detection (BN vs EN)

In [ ]:
def is_bangla(text):
    for ch in text:
        o = ord(ch) #unicode of bangla
        if 0x0980 <= o <= 0x09FF:
            return True
    return False    

In [ ]:
is_bangla("এই একটি বাংলা বাক্য।")

True

In [ ]:
is_bangla("সাদিয়া")

True

In [ ]:
is_bangla("this is an english sentence.")

False

## Load BN <-> EN translation model (MarianMT / OPUS MT)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

BN_EN_NAME = "Helsinki-NLP/opus-mt-bn-en"
EN_BN_NAME = "shhossain/opus-mt-en-to-bn"

tok_bn_en = MarianTokenizer.from_pretrained(BN_EN_NAME)
mod_bn_en = MarianMTModel.from_pretrained(BN_EN_NAME)

tok_en_bn = MarianTokenizer.from_pretrained(EN_BN_NAME)
mod_en_bn = MarianMTModel.from_pretrained(EN_BN_NAME)

def translate_bn_to_en(text):
    batch = tok_bn_en([text], return_tensors="pt",padding=True, truncation=True)
    gen = mod_bn_en.generate(**batch, max_new_tokens=128)
    return tok_bn_en.batch_decode(gen, skip_special_tokens=True)[0]


def translate_en_to_bn(text):
    batch = tok_en_bn([text], return_tensors="pt",padding=True, truncation=True)
    gen = mod_en_bn.generate(**batch, max_new_tokens=128)
    return tok_en_bn.batch_decode(gen, skip_special_tokens=True)[0]

print("Translation models loaded.")

Translation models loaded.


In [ ]:
print(translate_bn_to_en("বাংলাদেশ একটি সুন্দর দেশ।"))
print(translate_en_to_bn("Bangladesh is a beautiful country."))

Bangladesh is a beautiful country.
বাংলাদেশ একটি সুন্দর দেশ।


## 4.4 CLIR search function

In [ ]:
def clir_search(query, top_k=5):
    if is_bangla(query):
        q_en = translate_bn_to_en(query)
        results_bn = search_bn(query, top_k)
        results_en = search_en(q_en, top_k)
        return_en = {"queary_language": "bn", "translated_query": q_en, "results_language": "en", "results": results_en}
        return_bn = {"queary_language": "bn", "translated_query": q_en, "results_language": "bn", "results": results_bn}
        return return_bn,return_en
    else:
        q_bn = translate_en_to_bn(query)
        results_en = search_en(query, top_k)
        results_bn = search_bn(q_bn, top_k)
        return_bn = {"queary_language": "en", "translated_query": q_bn, "results_language": "bn", "results": results_bn}
        return_en = {"queary_language": "en", "translated_query": q_bn, "results_language": "en", "results": results_en}
        return return_bn,return_en

## 4.5 Test

In [ ]:
query_key= "বাংলাদেশ ক্রিকেট"
out = clir_search(query_key, top_k=5)


for r in out['results']:
    print(r['title'],"-", r['url'])
    print("____________________________________________")

Detected Bengali query. Translating to English for search...
Translated Query: Bangladesh Cricket
{'query_language': 'bn', 'translated_query': 'Bangladesh Cricket', 'results_language': 'en', 'results': [{'doc_id': 'en_000060', 'score': 7.069326487412225, 'title': 'bangladesh cricket board (bcb) has decided to start a new age-level tournament called rising stars u-23 cricket tournament, bcb officials informed after a board meeting on wednesday. the board meeting which started at 11 am, went on till evening before a media brief took place at 7:30 pm.', 'url': 'https://www.thedailystar.net/sports/cricket/news/bcb-introduce-new-age-group-u-23-tournament-4065501'}, {'doc_id': 'en_000116', 'score': 6.832835626336079, 'title': 'the bangladesh cricket board (bcb) yesterday named 28-year-old wicketkeeper-batter nurul hasan sohan captain for the t20i series in zimbabwe after deciding to rest regular skipper mahmudullah riyad for the t20is.', 'url': 'https://www.thedailystar.net/sports/cricket/ne

In [ ]:
query_key= "বাংলাদেশ ক্রিকেট"
out_bn, out_en = clir_search(query_key, top_k=5)

print("Print result in Bangla:")
for r in out_bn['results']:
    print(r['title'],"-", r['url'],r['score'])
    print("____________________________________________")


print("Result in English:")
for r in out_en['results']:
    print(r['title'],"-", r['url'],r['score'])
    print("____________________________________________")

Print result in Bangla:
ঢাকায় পৌঁছেছে শহীদ শরিফ ওসমান হাদির মরদেহ - https://bangladeshdiplomat.com/11050/latest/%e0%a6%a2%e0%a6%be%e0%a6%95%e0%a6%be%e0%a6%af%e0%a6%bc-%e0%a6%aa%e0%a7%8c%e0%a6%81%e0%a6%9b%e0%a7%87%e0%a6%9b%e0%a7%87-%e0%a6%b6%e0%a6%b9%e0%a7%80%e0%a6%a6-%e0%a6%b6%e0%a6%b0%e0%a6%bf%e0%a6%ab/ 1.7379079414714855
____________________________________________
সপরিবারে লন্ডন থেকে দেশের পথে তারেক রহমান - https://www.risingbd.com/politics/news/633345 1.7151871091754434
____________________________________________
স্বদেশ প্রত্যাবর্তনে তারেক রহমানকে স্বাগত জানিয়েছে ডিবিএ - https://www.risingbd.com/economics/stock-market/633295 1.6223062411528448
____________________________________________
‘বয়কট বাংলাদেশ’ পোস্টার শিলিগুড়ির হোটেল গাড়ি ট্যাক্সিতে - https://www.risingbd.com/international/news/633288 1.570339150955638
____________________________________________
বাংলাদেশ আর কখনো আধিপত্যবাদী শক্তির কাছে মাথা নত করবে না: মঞ্জু - https://www.risingbd.com/bangladesh/news/633337 1.534588

In [ ]:
out

{'query_language': 'en',
 'translated_query': 'বাংলাদেশ ক্রিকেট',
 'results_language': 'bn',
 'results': [{'doc_id': 'ba_000129',
   'score': 1.7379079414714855,
   'title': 'ঢাকায় পৌঁছেছে শহীদ শরিফ ওসমান হাদির মরদেহ',
   'url': 'https://bangladeshdiplomat.com/11050/latest/%e0%a6%a2%e0%a6%be%e0%a6%95%e0%a6%be%e0%a6%af%e0%a6%bc-%e0%a6%aa%e0%a7%8c%e0%a6%81%e0%a6%9b%e0%a7%87%e0%a6%9b%e0%a7%87-%e0%a6%b6%e0%a6%b9%e0%a7%80%e0%a6%a6-%e0%a6%b6%e0%a6%b0%e0%a6%bf%e0%a6%ab/'},
  {'doc_id': 'ba_000033',
   'score': 1.7151871091754434,
   'title': 'সপরিবারে লন্ডন থেকে দেশের পথে তারেক রহমান',
   'url': 'https://www.risingbd.com/politics/news/633345'},
  {'doc_id': 'ba_000083',
   'score': 1.6223062411528448,
   'title': 'স্বদেশ প্রত্যাবর্তনে তারেক রহমানকে স্বাগত জানিয়েছে ডিবিএ',
   'url': 'https://www.risingbd.com/economics/stock-market/633295'},
  {'doc_id': 'ba_000090',
   'score': 1.570339150955638,
   'title': '‘বয়কট বাংলাদেশ’ পোস্টার শিলিগুড়ির হোটেল গাড়ি ট্যাক্সিতে',
   'url': 'https://www.